# 03 - Train Sentiment Model

Trains a transparent baseline for student-feedback sentiment classification. It selects hyperparameters on the training split, reports validation and untouched-test performance, and saves one deployable scikit-learn pipeline.

In [ ]:
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
AI_DIR = Path.cwd().resolve().parent
PROCESSED_DIR = AI_DIR / 'datasets' / 'processed'
MODEL_DIR = AI_DIR / 'models' / 'sentiment'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def load_split(name: str) -> pd.DataFrame:
    path = PROCESSED_DIR / f'sentiment_{name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'{path} does not exist. Run 02_preprocess_sentiment.ipynb first.')
    return pd.read_csv(path)

train_df = load_split('train')
validation_df = load_split('validation')
test_df = load_split('test')
print({name: len(split) for name, split in {'train': train_df, 'validation': validation_df, 'test': test_df}.items()})

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, strip_accents='unicode')),
    ('classifier', LogisticRegression(max_iter=2_000, class_weight='balanced', random_state=RANDOM_STATE)),
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 2],
    'classifier__C': [0.5, 1.0, 2.0],
}

search = GridSearchCV(
    estimator=pipeline, param_grid=param_grid, scoring='f1_macro', cv=5, n_jobs=-1, refit=True
)
search.fit(train_df['feedback_text'], train_df['sentiment'])
best_model = search.best_estimator_
print(f'Best CV macro F1: {search.best_score_:.3f}')
print(f'Best parameters: {search.best_params_}')

In [ ]:
def evaluate(split_name: str, split_df: pd.DataFrame):
    predictions = best_model.predict(split_df['feedback_text'])
    macro_f1 = f1_score(split_df['sentiment'], predictions, average='macro')
    print(f'{split_name.title()} macro F1: {macro_f1:.3f}')
    print(classification_report(split_df['sentiment'], predictions, digits=3))
    ConfusionMatrixDisplay.from_predictions(split_df['sentiment'], predictions, cmap='Blues', xticks_rotation=0)
    return macro_f1, classification_report(split_df['sentiment'], predictions, output_dict=True)

validation_f1, validation_report = evaluate('validation', validation_df)

In [ ]:
# The test split is evaluated once after model selection.
test_f1, test_report = evaluate('test', test_df)

model_path = MODEL_DIR / 'sentiment_tfidf_logreg.joblib'
metrics_path = MODEL_DIR / 'sentiment_tfidf_logreg_metrics.json'
joblib.dump(best_model, model_path)

metrics = {
    'model': 'TF-IDF + LogisticRegression',
    'random_state': RANDOM_STATE,
    'best_cv_macro_f1': search.best_score_,
    'best_params': search.best_params_,
    'validation_macro_f1': validation_f1,
    'test_macro_f1': test_f1,
    'validation_report': validation_report,
    'test_report': test_report,
}
metrics_path.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print(f'Saved model: {model_path}')
print(f'Saved metrics: {metrics_path}')

In [ ]:
examples = pd.Series([
    'The instructor explained difficult concepts clearly and the labs were useful.',
    'The deadlines were unreasonable and the feedback was not helpful.',
    'The course was acceptable, but nothing stood out.',
])
pd.DataFrame({'feedback_text': examples, 'predicted_sentiment': best_model.predict(examples)})